# SETUP

In [ ]:
import imageio.v3 as iio
import numpy as np
import os
import pandas as pd
import skimage
import warnings

from datetime import datetime
from pathlib import Path
from scipy.ndimage import gaussian_filter
warnings.filterwarnings("ignore")

import resource

## INPUT

In [ ]:
dir_input = "INPUT FOLDER"
filetype = "tif"

### PREFERENCES

In [ ]:
normalize = True
saturated_pct = 0.001

## INPUT CHECK

In [ ]:
if type(dir_input) != str:
    raise TypeError("dir_input must be a string.")

if type(filetype) != str:
    raise TypeError("filetype must be a string.")

if type(normalize) != bool:
    normalize = True
    print("normalize must be set to either True or False; using default value (True).")

if normalize:
    norm_status = "normalized"
    saturated_pct = saturated_pct
    try:
        saturated_pct = float(saturated_pct)
        if not 0 <= saturated_pct <= 100:
            raise ValueError()
        if saturated_pct > 1:
            print("Caution: setting a high value for saturated_pct could result in the loss of important data.")
    except:
        saturated_pct = 0.001
        print("saturated_pct must be a number between 0-100; using default value (0.001).")
    saturated_pct_text = str(saturated_pct).replace(".", "pt")
    norm_status = f'{norm_status}, {saturated_pct_text} percent saturation'
else:
    norm_status = "no normalization"
    saturated_pct = None

now = datetime.now() 
dt_string = now.strftime("%Y-%m-%d_%H%M%S")
dir_output = Path("output", f'{dt_string} - {dir_input} - image pre-processing ({norm_status})')

# FUNCTIONS

In [ ]:
def filter_hidden(some_list):
    new_list = sorted([x for x in some_list if "._" not in x])
    if len(new_list) < len(some_list):
        print(f'Detected {len(some_list) - len(new_list)} hidden files in the input directory\n'
              + f'Hidden image files have been excluded from analysis.')
    return(new_list)

In [ ]:
def summary():
    analysis_summary = f'Analysis parameters...\n\n'
    analysis_summary += f'Normalization: {normalize}\n'
    if normalize:
        analysis_summary += f'Saturated percent: {saturated_pct}'
    summary = open(Path(dir_output, f'Analysis parameters for {dir_input} image pre-processing.txt'), "w")
    summary.write(analysis_summary)
    summary.close()

# FILE PARSE

In [ ]:
image_inputs = [str(path) for path in Path(dir_input).rglob(f'*.{filetype}')]
image_files = filter_hidden(image_inputs)
image_details = [Path(file_name).stem.split('_') for file_name in image_files]

area_inputs = [str(path) for path in Path(dir_input).rglob(f'*.csv')]
area_files = filter_hidden(area_inputs)
area_details = [Path(file_name).stem.split('_') for file_name in area_files]

# MAIN

In [ ]:
for item in area_details:
    matching_images = [x for x in image_details if set(item).issubset(x)]
    area_pct = pd.read_csv(Path(dir_input, f'{"_".join(item)}.csv'))['Math_SpheroidArea'][0]
    bg_percentile = 100 - area_pct
    if bg_percentile < 0.01:
        bg_percentile = 0.01
    for matching_image in matching_images:
        image_filename = (f'{"_".join(matching_image)}')
        image_path = Path(dir_input, f'{image_filename}.{filetype}')
        df_image = pd.DataFrame(skimage.io.imread(image_path, as_gray = True)).astype("float64")
        df_image = pd.DataFrame(gaussian_filter(df_image, sigma = 1))
        empty_bg = np.percentile(df_image, bg_percentile)
        df_image_temp = (df_image - empty_bg).clip(lower = 0)
        thresh = skimage.filters.threshold_otsu(df_image_temp.values)
        bg = ((empty_bg * bg_percentile) + (thresh * area_pct)) / 100
        if empty_bg >= bg:
            bg = empty_bg
        df_image_bgsub = (df_image - bg).clip(lower = 0)
        if normalize:
            sat = np.percentile(df_image_bgsub, (100 - saturated_pct))
            df_image_bgsub = (df_image_bgsub * (65535 / sat)).clip(lower = 0, upper = 65535)
            image_filename_bgsub = f'{image_filename}_bgsub(norm{saturated_pct_text}).{filetype}'
        else:
            image_filename_bgsub = f'{image_filename}_bgsub(nonorm).{filetype}'
        df_image_bgsub = df_image_bgsub.astype("uint16")
        array_image_bgsub = np.asarray(df_image_bgsub)
        dir_image_out = Path(dir_output, image_filename_bgsub)
        if not os.path.exists(dir_output):
            os.makedirs(dir_output)
        print(f'Writing {image_filename_bgsub}...')
        iio.imwrite(dir_image_out, array_image_bgsub)
summary()

In [ ]:
now = datetime.now()
print ("\n**********SCRIPT COMPLETE**********")
print(now.strftime("%Y-%m-%d_%H%M%S"))